<a href="https://colab.research.google.com/github/sandrokhizanishvili/AML_GNN_GMA/blob/main/model_pna_enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GATv2 Baseline for Anti-Money Laundering Detection

**Course:** Graph Mining and Applications, Sapienza University of Rome

---

## What This Notebook Does

This notebook trains a **GATv2 baseline** without GFP features, using only the
original 16-dimensional edge features. It is the ablation counterpart to
`gat-gfp.ipynb`, isolating the contribution of GFP features to GATv2 performance.

| Property | Value |
|----------|-------|
| Base architecture | GATv2Conv + edge readout |
| Node feature dims | 5 (original, no GFP) |
| Edge feature dims | 16 (baseline only, no GFP) |
| Attention heads | 4 |


## 1. Installation

This cell installs the correct versions of PyTorch and PyTorch Geometric for Colab.


If the libraries are already installed correctly, you can skip this cell.

In [1]:
print("🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...")

# 1. Uninstall the current mismatching versions
# We remove the one you just spent 15 mins compiling, because re-installing
# the CORRECT version via wheels will take only 30 seconds.
# os.system("pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib")
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib

# 2. Install PyTorch 2.8.0 (with CUDA 12.6 support)
# We specify the version explicitly to match the PyG documentation you found.
print("⬇️ Installing PyTorch 2.8.0...")
# os.system("pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# 3. Install Graph Libraries for PyTorch 2.8
# This link matches the table in your screenshot: torch-2.8.0 + cu126
print("⬇️ Installing Graph Libraries (Wheels)...")
# os.system("pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html")
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

print("="*40)
print("✅ SETUP COMPLETE.")
print("⚠️ YOU MUST RESTART THE RUNTIME NOW (Runtime -> Restart Session)")
print("="*40)




import torch

try:
    import torch_sparse
    sparse_status = "✅ Installed"
    sparse_version = torch_sparse.__version__
except ImportError:
    sparse_status = "❌ Not Found"
    sparse_version = "N/A"

print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
print(f"Torch Sparse Status:  {sparse_status} ({sparse_version})")

if torch.cuda.is_available() and sparse_status == "✅ Installed":
    print("\nSUCCESS! You are ready to run the training loop.")
else:
    print("\n⚠️ Something is still missing. Did you Restart the Runtime?")



!pip install torch_geometric

🚀 SWITCHING TO PYTORCH 2.8 (FAST MODE)...
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
⬇️ Installing PyTorch 2.8.0...
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 174.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 200.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 134.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 85.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 100.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/

## 2. Imports and Reproducibility

We import all required libraries and fix all random seeds so that results are reproducible across runs.

The main libraries used are:
- **PyTorch** for model definition and training
- **PyTorch Geometric (PyG)** for graph data structures and GNN layers
- **scikit-learn** for evaluation metrics
- **tqdm** for progress bars during training

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import time
import math
import random
import warnings
from google.colab import drive
warnings.filterwarnings('ignore')

# ── Numerical ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, precision_recall_curve, average_precision_score, matthews_corrcoef, auc
)

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── PyTorch Geometric ─────────────────────────────────────────────────────────
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import degree

# ── Progress bar ──────────────────────────────────────────────────────────────
from tqdm import tqdm

print(f'PyTorch          : {torch.__version__}')
print(f'PyTorch Geometric: {torch_geometric.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PyTorch          : 2.8.0+cu126
PyTorch Geometric: 2.7.0
Device           : cuda


## 3. Loading the Graph Data

The graph was built in `Data_prepration.ipynb` and saved as three PyG `Data` objects
using the **baseline feature set** (16 edge features, 5 node features). We load them here.

### Why Three Separate Graphs?

We use a **cumulative snapshot** design that follows the paper's protocol:

- `train_graph` contains only the training period edges. All of them are labelled and used for training.
- `val_graph` contains training + validation period edges. Only the new validation edges are evaluated; the training edges provide neighbourhood context for message passing.
- `test_graph` contains all edges from all periods. Only the test period edges are evaluated.

This design is important because financial transactions exist in a temporal context. A suspicious transaction in the test set should be evaluated using knowledge of the account's full history, not just the test period. By including all prior edges in each snapshot, we give the GNN access to that historical context during message passing.

### Data Split (Temporal)

The split is done chronologically to prevent data leakage:
- **Train:** first 60% of time period
- **Validation:** next 20%
- **Test:** final 20%

In [3]:
# # Mount Google Drive where the graph files and model checkpoints are stored

# drive.mount('/content/drive')

# os.chdir('/content/drive/MyDrive/GMA_GNN_AML')

In [4]:
DATA_DIR = '/kaggle/input/datasets/sandrokh/aml-gnn-pna'

# Use baseline graphs (no GFP features)
train_graph = torch.load(os.path.join(DATA_DIR, 'train_graph.pt'), weights_only=False)
val_graph   = torch.load(os.path.join(DATA_DIR, 'val_graph.pt'),   weights_only=False)
test_graph  = torch.load(os.path.join(DATA_DIR, 'test_graph.pt'),  weights_only=False)

def describe_graph(g, name):
    labels = g.y[g.eval_mask]
    n_pos  = (labels == 1).sum().item()
    n_eval = g.eval_mask.sum().item()
    print(f'{name}: nodes={g.num_nodes:,}  edges={g.edge_index.shape[1]:,}  '
          f'eval={n_eval:,}  laund={n_pos:,} ({100*n_pos/n_eval:.4f}%)  '
          f'node_dim={g.x.shape[1]}  edge_dim={g.edge_attr.shape[1]}')

describe_graph(train_graph, 'train_graph')
describe_graph(val_graph,   'val_graph')
describe_graph(test_graph,  'test_graph')

for g, name in [(train_graph,'train'),(val_graph,'val'),(test_graph,'test')]:
    assert (g.y[g.eval_mask] == -1).sum() == 0
print('Label sanity check passed.')


train_graph: nodes=712,684  edges=4,154,429  eval=4,154,429  laund=1,813 (0.0436%)  node_dim=5  edge_dim=16
val_graph: nodes=712,684  edges=5,539,239  eval=1,384,810  laund=827 (0.0597%)  node_dim=5  edge_dim=16
test_graph: nodes=712,684  edges=6,924,049  eval=1,384,810  laund=925 (0.0668%)  node_dim=5  edge_dim=16
Label sanity check passed.


## 4. Hyperparameters

All training and model hyperparameters are defined here in one place for easy tuning.

### Alignment with the Paper

The following settings match the paper (Appendix E, Table 11, Table 12):
- `NUM_LAYERS = 2` -- number of GNN message passing layers
- `NUM_NEIGHBORS = [100, 100]` -- 100 one-hop and 100 two-hop neighbours sampled per seed edge

### GAT-Specific Hyperparameters

- `NUM_HEADS = 4` -- number of attention heads per GATv2 layer. Each head learns an independent attention function, allowing the model to simultaneously attend to different aspects of the neighbourhood (e.g. one head for amount-based attention, another for timing-based attention). We concatenate the heads (`concat=True`), setting `out_channels = hidden_dim // num_heads = 16` per head so the concatenated output remains at `hidden_dim = 64`.
- `HIDDEN_DIM = 64` -- matches the paper's baseline hidden size. The edge projection maps 16 baseline features to `hidden_dim` (16 → 64), giving the model sufficient representational capacity without unnecessary expansion.
- `EDGE_DIM = 16` -- the 16 baseline transaction features (amount, currency, payment format, etc.) are used as edge attributes. This notebook uses no GFP features and no RWPE, making it the pure GATv2 baseline. The attention mechanism uses these features to compute per-edge importance scores.

### Class Imbalance Handling

With a 2,290:1 ratio of legitimate to laundering transactions, the model would simply predict everything as legitimate without correction. We use `pos_weight = 8` in the Binary Cross-Entropy loss, which means every laundering transaction contributes 8x more to the loss than a legitimate one. This forces the model to pay attention to the rare class.

The paper used `pos_weight` in the range (6, 8) for the GNN baselines (Table 11). We use 8 as the upper bound of this range.

In [5]:
NODE_DIM = train_graph.x.shape[1]          # 5 (no GFP, no RWPE)
EDGE_DIM = train_graph.edge_attr.shape[1]  # 16 (baseline only)

# HIDDEN_DIM=64 (vs 128 in gat-gfp) since 16-dim edges don't need larger projection
HIDDEN_DIM    = 64
NUM_LAYERS    = 2
NUM_HEADS     = 4
DROPOUT       = 0.3
EPOCHS        = 20
LR            = 1e-3
WEIGHT_DECAY  = 1e-5
NUM_NEIGHBORS = [100, 100]
BATCH_SIZE    = 2048

n_neg = (train_graph.y[train_graph.eval_mask] == 0).sum().item()
n_pos = (train_graph.y[train_graph.eval_mask] == 1).sum().item()
print(f'Train imbalance: {n_neg/n_pos:.0f}:1')

POS_WEIGHT = torch.tensor([8.0], device=device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
print(f'NODE_DIM={NODE_DIM}  EDGE_DIM={EDGE_DIM}  HIDDEN_DIM={HIDDEN_DIM}  EPOCHS={EPOCHS}')


Train imbalance: 2290:1
NODE_DIM=5  EDGE_DIM=16  HIDDEN_DIM=64  EPOCHS=20


## 5. Model Architecture

### The Core Problem: Classifying Edges, Not Nodes

Standard GNNs produce **node embeddings** through message passing. But our task is to classify **edges** (transactions). This requires a two-step pattern:

1. **Encode** -- run message passing over the neighbourhood graph to build informative node embeddings
2. **Decode** -- for each transaction we want to classify, look up the sender and receiver embeddings and combine them with the transaction's own features to produce a classification

This encode-decode separation is also required by how PyG's `LinkNeighborLoader` works. It keeps two sets of edges in each batch:
- `batch.edge_index` -- context edges used only for message passing (no labels)
- `batch.edge_label_index` -- seed edges that we actually want to classify (have labels)

### Why LayerNorm Instead of BatchNorm

BatchNorm normalises across the entire batch. With many context edges per batch but very few laundering edges on average, the batch mean and variance are completely dominated by legitimate transactions. LayerNorm normalises each node independently across its 64 features, so laundering nodes keep their distinctive activation patterns regardless of the class distribution in the batch.

### GATv2 Architecture Overview

GATv2 was introduced by Brody et al. (2022) as a fix to the original GAT. The original GAT computes attention as:

```
GAT:   e(i,j) = a · LeakyReLU( W·h_i || W·h_j )   -- static attention
GATv2: e(i,j) = a · LeakyReLU( W · [h_i || h_j] ) -- dynamic attention
```

The difference is subtle but important: GAT's attention is "static" because `W·h_i` is computed independently of `h_j`, meaning the ranking of neighbours for node i is the same regardless of i's own representation. GATv2 fixes this by applying the linear transformation after concatenation, making attention truly dynamic — the importance of neighbour j depends on both j and i together.

With edge features (the `GATv2Conv` implementation in PyG), the attention score also incorporates the edge feature `e_ij`:

```
e(i,j) = a · LeakyReLU( W · [h_i || h_j || e_ij] )
```

This is exactly what we want for AML — the attention score uses the transaction features (amount, currency, payment format) to decide how much weight to give each neighbour when building a node's embedding.

The full forward pass for one transaction A to B looks like this:

```
INPUT
  x [N, 5]                      -- raw baseline node features for all accounts
  edge_attr [E, 16]             -- baseline edge features for context transactions
  edge_label_attr [n_seeds, 16] -- baseline features of the seed transactions

ENCODE (message passing with attention)
  node_proj:  x [N, 5]            --> h0 [N, 64]
  edge_proj:  edge_attr [E, 16]   --> e  [E, 64]   (expanding: 16 → 64)

  Layer 1 (GATv2Conv, 4 heads concatenated):
    For each account v and each neighbour u:
      attention(v,u) = softmax_u( a · LeakyReLU( W · [h0_v || h0_u || e_vu] ) )
    h1_v = concat_heads( sum_u( attention(v,u) · W · h0_u ) )  -- 4 heads × 16 = 64
    Then: LayerNorm -> ReLU -> Dropout

  Layer 2 (GATv2Conv): same as layer 1, using h1
    Produces h2 [N, 64]

DECODE
  e_seed  = edge_proj(edge_label_attr)              --> [n_seeds, 64]
  edge_emb = concat(h2[A], h2[B], e_seed)           --> [n_seeds, 192]
  logit    = MLP(edge_emb)  -- 192 -> 64 -> 1
  P(laundering) = sigmoid(logit)
```

The three components in the decoder each carry distinct information:
- `h2[A]` -- what kind of sender account A is (2-hop attention-weighted neighbourhood context)
- `h2[B]` -- what kind of receiver account B is (2-hop attention-weighted neighbourhood context)
- `e_seed` -- this specific transaction's baseline features (amount, currency, payment format)

### Multi-Head Attention

We use 4 attention heads with `concat=True`, meaning the outputs of all 4 heads are **concatenated** rather than averaged. Each head produces a 16-dimensional output (`out_channels = hidden_dim // num_heads = 64 // 4 = 16`), and concatenating them gives `4 × 16 = 64 = hidden_dim`. This keeps the node embedding dimension consistent across layers.

Each head learns an independent attention function and can specialise on different neighbourhood signals — for example, one head might learn to attend to neighbours with large amounts, another to neighbours with the same currency, and another to neighbours with similar transaction timing.

In [ ]:
def build_mlp(in_dim, hidden_dim, out_dim, num_layers=2, dropout=0.3):
    """
    Build a multi-layer perceptron (MLP) with LayerNorm and Dropout.

    This function is used as the final edge classifier in the decoder.

    The last linear layer has no activation or normalisation, because
    the output is either fed into the next layer (which has its own activation)
    or is a raw logit that will be passed to sigmoid/BCE loss.

    Parameters
    ----------
    in_dim     : input feature dimension
    hidden_dim : width of intermediate layers
    out_dim    : output feature dimension
    num_layers : total number of linear layers
    dropout    : fraction of features randomly zeroed during training

    Returns
    -------
    nn.Sequential : the constructed MLP
    """
    layers = []
    dims   = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]

    for i in range(len(dims) - 1):
        layers.append(nn.Linear(dims[i], dims[i+1]))
        # Add activation, normalisation, and dropout after every layer except the last
        if i < len(dims) - 2:
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(dims[i+1]))
            layers.append(nn.Dropout(dropout))

    return nn.Sequential(*layers)

In [ ]:
class GATModel(nn.Module):
    """
    Graph Attention Network v2 (GATv2) for edge classification.

    Uses GATv2Conv layers with multi-head attention for message passing,
    followed by an edge readout decoder that combines sender/receiver
    node embeddings with the seed transaction's edge features.

    GATv2 differs from GAT in that it computes dynamic attention —
    the importance of neighbour j truly depends on both i and j together,
    rather than being a static function of j alone. With edge features,
    the attention score also incorporates the transaction features (amount,
    currency, payment format), making each neighbour's importance
    dependent on the specific transaction connecting them.

    The 16 baseline edge features are projected through a single
    Linear(16 → 64). The same projection is shared between encode()
    (message passing) and decode() (seed edge classification).

    Parameters
    ----------
    node_dim   : number of raw node features (5)
    edge_dim   : number of edge features (16 baseline)
    hidden_dim : size of all hidden embeddings (64)
    num_layers : number of GATv2 message passing layers (2)
    num_heads  : number of attention heads per layer (4)
    dropout    : dropout rate applied after each GNN layer
    """

    def __init__(self, node_dim, edge_dim, hidden_dim, num_layers, num_heads, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        # Node projection: raw node features → hidden space
        self.node_proj = nn.Linear(node_dim, hidden_dim)

        # Edge projection: 16 baseline features → hidden space (expanding: 16 → 64)
        # This is used in both encode() for message passing and decode() for seed features.
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GATv2Conv(
                in_channels  = hidden_dim,
                out_channels = hidden_dim // num_heads,  # 16 per head
                heads        = num_heads,   # 4 heads concatenated → 64
                concat       = True,        # concatenate heads: 4 * 16 = 64 = hidden_dim
                edge_dim     = hidden_dim,  # projected edge features
                dropout      = dropout,
                add_self_loops = False,     # self-loops not meaningful for transaction edges
            ))
            self.norms.append(nn.LayerNorm(hidden_dim))

        # Edge classifier: concat(h[src], h[dst], e_seed) -> 1
        # Input dim = hidden_dim * 3  (64 * 3 = 192)
        self.edge_classifier = build_mlp(
            in_dim     = hidden_dim * 3,  
            hidden_dim = hidden_dim,
            out_dim    = 1,
            dropout    = dropout,
        )

    def encode(self, x, edge_index, edge_attr):
        """
        Run GATv2 message passing over the context graph.

        The 16 baseline edge features are projected and fed into the attention
        mechanism, so transaction-level features (amount, currency, payment
        format) inform which neighbours each account attends to.

        Parameters
        ----------
        x          : raw node features [N, node_dim]
        edge_index : context edge connectivity [2, E]
        edge_attr  : context edge features [E, edge_dim]

        Returns
        -------
        h : node embeddings after all GATv2 layers [N, hidden_dim]
        """
        h = F.relu(self.node_proj(x))
        e = F.relu(self.edge_proj(edge_attr))  # [E, 64]

        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index, edge_attr=e)
            h = norm(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h

    def decode(self, h, edge_label_index, edge_label_attr):
        """
        Classify seed edges using three sources of information:
          - h[src]  : sender account's 2-hop attention-weighted context
          - h[dst]  : receiver account's 2-hop attention-weighted context
          - e_seed  : this transaction's baseline features (amount, currency, payment format)

        Parameters
        ----------
        h                : node embeddings from encode() [N, hidden_dim]
        edge_label_index : seed edge endpoints [2, n_seeds]
        edge_label_attr  : baseline features of seed edges [n_seeds, edge_dim]

        Returns
        -------
        logits : classification scores [n_seeds] (before sigmoid)
        """
        src, dst = edge_label_index

        e_seed   = F.relu(self.edge_proj(edge_label_attr))            # [n_seeds, 64]
        edge_emb = torch.cat([h[src], h[dst], e_seed], dim=-1)        # [n_seeds, 192]

        return self.edge_classifier(edge_emb).squeeze(-1)

    def forward(self, x, edge_index, edge_attr, edge_label_index, edge_label_attr):
        """
        Full forward pass: encode the context graph with attention-weighted
        message passing, then decode the seed edges.

        Parameters
        ----------
        x                : node features [N, node_dim]
        edge_index       : context edge connectivity [2, E]
        edge_attr        : context edge features [E, edge_dim]
        edge_label_index : seed edge endpoints [2, n_seeds]
        edge_label_attr  : seed edge features [n_seeds, edge_dim]

        Returns
        -------
        logits : classification scores [n_seeds]
        """
        h = self.encode(x, edge_index, edge_attr)
        return self.decode(h, edge_label_index, edge_label_attr)

## 6. Data Loader

### Why We Cannot Load the Full Graph at Once

The training graph has over 4 million edges. Loading all of them for a single forward pass would require several gigabytes of GPU memory just for the activations and gradients. Instead, we use PyG's `LinkNeighborLoader` to process the graph in mini-batches.

For each mini-batch, the loader:
1. Picks a batch of seed edges (the transactions we want to classify)
2. For each seed edge's endpoints, samples a local neighbourhood (100 one-hop + 100 two-hop neighbours)
3. Builds a small subgraph from those sampled neighbours
4. Returns the subgraph with two edge sets:
   - `batch.edge_index`: the neighbourhood context edges (used for message passing)
   - `batch.edge_label_index`: the seed edges (used for classification and loss)

### How Seed Edge Features Are Tracked

`make_loader` returns both the loader and `seed_edge_attr` (the raw features for all seed edges). Inside each batch, `batch.input_id` tells us which seed edges from the full pool ended up in this batch, so we can fetch the correct features with `seed_edge_attr[batch.input_id.cpu()]`.

### Class Imbalance in Batches

With 0.0436% laundering rate and batch size 2,048, each batch sees on average fewer than 1 laundering transaction on average. The `pos_weight=8` in the loss function compensates for this by amplifying the gradient from those rare edges.

In [ ]:
def make_loader(graph, shuffle=True, verbose=False):
    """
    Build a LinkNeighborLoader for mini-batch edge classification.

    The loader uses the real class distribution (no oversampling). Class
    imbalance is handled instead through pos_weight in the loss function.

    Returns a tuple of (loader, seed_edge_attr) because the loader itself
    does not carry seed edge features -- they must be looked up separately
    using batch.input_id in the training loop.

    Parameters
    ----------
    graph   : PyG Data object (train_graph, val_graph, or test_graph)
    shuffle : True during training for randomness; False during evaluation
    verbose : if True, print the number of positive and negative seed edges

    Returns
    -------
    loader         : LinkNeighborLoader that yields mini-batches
    seed_edge_attr : raw edge features for all seed edges [n_seeds, 16]
    """
    # Extract only the labelled edges from this graph snapshot.
    # eval_mask marks which edges belong to this split's evaluation set.
    seed_mask       = graph.eval_mask
    seed_edge_index = graph.edge_index[:, seed_mask]  # [2, n_seeds]
    seed_labels     = graph.y[seed_mask].float()       # [n_seeds] -- 0.0 or 1.0
    seed_edge_attr  = graph.edge_attr[seed_mask]       # [n_seeds, 16] -- returned separately

    if verbose:
        n_pos = (seed_labels == 1).sum().item()
        n_neg = (seed_labels == 0).sum().item()
        print(f'  Seed edges : {n_pos + n_neg:,} total')
        print(f'  Laundering : {n_pos:,} ({100*n_pos/(n_pos+n_neg):.4f}%)')

    loader = LinkNeighborLoader(
        data             = graph,           # full graph (for neighbourhood sampling)
        num_neighbors    = NUM_NEIGHBORS,   # [100, 100] matches paper
        edge_label_index = seed_edge_index, # which edges to classify
        edge_label       = seed_labels,     # their labels (0 or 1)
        batch_size       = BATCH_SIZE,      # seed edges per mini-batch
        shuffle          = shuffle,
        num_workers      = 0,               # must be 0 in Colab (multiprocessing issues)
        pin_memory       = False,
    )

    return loader, seed_edge_attr

## 7. Training and Evaluation

### Training Loop (`train_epoch`)

Each epoch iterates over all mini-batches produced by the loader. For each batch:
1. The context subgraph is encoded to produce node embeddings
2. The seed edge features are fetched using `batch.input_id` as an index into `seed_edge_attr`
3. The decoder classifies each seed edge using sender embedding + receiver embedding + edge features
4. BCE loss is computed with `pos_weight=8` to upweight laundering edges
5. Gradients are clipped to prevent instability, then weights are updated

### Evaluation Loop (`evaluate`)

Evaluation collects predicted probabilities and true labels across all batches, then computes metrics at a fixed threshold. The threshold default is 0.5, but this is not optimal for imbalanced data -- threshold tuning is done separately in Section 9.

### Checkpointing Strategy

We checkpoint based on **validation PR-AUC**. PR-AUC is threshold-independent and more informative than ROC-AUC for severely imbalanced datasets, because it focuses on the precision-recall tradeoff for the minority class.

### Learning Rate Schedule

Cosine annealing smoothly reduces the learning rate from the initial value down to `eta_min=1e-5` over the training run. This prevents overshooting at the end of training and typically improves final performance.

In [ ]:
def train_epoch(model, graph, optimizer):
    """
    Run one full training epoch over all seed edges in the graph.

    Iterates over mini-batches from the loader. For each batch, performs
    a forward pass, computes loss, and updates model weights.

    Parameters
    ----------
    model     : GATv2 model instance
    graph     : training graph (train_graph)
    optimizer : Adam optimiser

    Returns
    -------
    float : average BCE loss across all batches in this epoch
    """
    model.train()
    loader, seed_edge_attr = make_loader(graph, shuffle=True, verbose=True)
    total_loss = 0.0
    n_batches  = 0
    pbar       = tqdm(loader, desc='  Training', leave=False)

    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Fetch the raw edge features for the seed edges in this batch.
        # batch.input_id contains the positions of this batch's seed edges
        # in the full seed pool returned by make_loader.
        # We index on CPU then move to GPU to avoid device mismatch errors.
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        # Forward pass: encode context graph, decode seed edges
        logits = model(
            batch.x,
            batch.edge_index,        # context edges for message passing
            batch.edge_attr,         # context edge features
            batch.edge_label_index,  # seed edges to classify
            seed_attr,               # seed edge features
        )

        # Compute loss against true labels (0=legitimate, 1=laundering)
        loss = criterion(logits, batch.edge_label)
        loss.backward()

        # Clip gradients to prevent exploding gradients on the sparse laundering signal
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, graph, threshold=0.5):
    """
    Evaluate the model on a graph snapshot and return classification metrics.

    Uses the real class distribution (no oversampling) so that metrics
    reflect true performance on the original data. The threshold parameter
    controls the boundary between predicted laundering and legitimate.

    Note: threshold=0.5 is used here for monitoring during training.
    The optimal threshold is found separately using find_best_threshold()
    after training completes.

    Parameters
    ----------
    model     : trained GATv2 model
    graph     : graph to evaluate on (val_graph or test_graph)
    threshold : decision boundary for converting probabilities to predictions

    Returns
    -------
    dict with keys: f1, precision, recall, roc_auc, pr_auc
    """
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)
    all_probs  = []
    all_labels = []
    pbar       = tqdm(loader, desc='  Validation', leave=False)

    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        # Convert logits to probabilities and collect across batches
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())

    # Concatenate all batches into single arrays
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # Apply threshold to get binary predictions
    preds = (all_probs >= threshold).astype(int)

    # Compute minority-class metrics (pos_label=1 means we evaluate on laundering class only)
    f1  = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    pre = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
        precision_curve, recall_curve, _ = precision_recall_curve(all_labels, all_probs)
        pr_auc = auc(recall_curve, precision_curve)
    except ValueError:
        # This can happen if a batch has no positives -- safe fallback
        roc_auc = float('nan')
        pr_auc  = float('nan')

    return {'f1': f1, 'precision': pre, 'recall': rec, 'roc_auc': roc_auc, 'pr_auc': pr_auc}


def run_training(model, model_name, checkpoint_path, epochs=EPOCHS):
    """
    Full training loop with validation monitoring and model checkpointing.

    Trains the model for the specified number of epochs, evaluating on the
    validation set after each epoch. Saves the model state whenever validation
    PR-AUC improves. At the end, loads the best checkpoint and evaluates on the
    test set.

    We checkpoint on PR-AUC rather than F1 because PR-AUC is threshold-independent
    and more reliable when F1 at threshold=0.5 is often 0.0 early in training
    (the model outputs low probabilities before it has learned to discriminate).

    Parameters
    ----------
    model           : GATv2 model instance
    model_name      : name string used in printed output
    checkpoint_path : file path to save the best model weights
    epochs          : number of training epochs

    Returns
    -------
    model        : model loaded with best checkpoint weights
    history      : list of dicts with per-epoch metrics
    test_metrics : dict with final test set results
    """
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Cosine annealing smoothly reduces learning rate to eta_min over all epochs.
    # This avoids overshooting at the end of training.
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_val_pr_auc = 0.0
    best_state      = None
    history         = []

    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
    print(f'  pos_weight : {float(POS_WEIGHT)} (within paper range 6-8)')
    print(f'{"="*60}')

    for epoch in range(1, epochs + 1):
        print(f'\n--- Epoch {epoch}/{epochs} ---')
        t0 = time.time()

        # Training step
        train_loss = train_epoch(model, train_graph, optimizer)
        scheduler.step()  # update the learning rate for the next epoch

        # Validation step
        val_metrics = evaluate(model, val_graph)
        elapsed     = time.time() - t0

        history.append({'epoch': epoch, 'loss': train_loss, **val_metrics})

        # Save model if validation PR-AUC improved
        improved = ''
        if val_metrics['pr_auc'] > best_val_pr_auc:
            best_val_pr_auc = val_metrics['pr_auc']
            # Deep copy the state so future epochs do not overwrite it
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}
            os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
            torch.save(model.state_dict(), checkpoint_path)
            improved = '  --> New Best Model!'

        print(
            f'Result: Train Loss: {train_loss:.4f} | '
            f'Val F1: {val_metrics["f1"]:.4f} | '
            f'Val Pre: {val_metrics["precision"]:.4f} | '
            f'Val Rec: {val_metrics["recall"]:.4f} | '
            f'Val ROC-AUC: {val_metrics["roc_auc"]:.4f} | '
            f'Val PR-AUC: {val_metrics["pr_auc"]:.4f} | '
            f'Time: {elapsed:.1f}s'
            f'{improved}'
        )

    # Restore best checkpoint for final evaluation
    if best_state is not None:
        model.load_state_dict(best_state)

    # Final evaluation on the held-out test set
    test_metrics = evaluate(model, test_graph)
    print(f'\n{"="*60}')
    print(f'Final Test Results -- {model_name}')
    print(f'  F1        : {test_metrics["f1"]:.4f}')
    print(f'  Precision : {test_metrics["precision"]:.4f}')
    print(f'  Recall    : {test_metrics["recall"]:.4f}')
    print(f'  ROC-AUC   : {test_metrics["roc_auc"]:.4f}')
    print(f'  PR-AUC    : {test_metrics["pr_auc"]:.4f}')
    print(f'{"="*60}')

    return model, history, test_metrics

## 8. Training the Model

We initialise GATv2 and train it for 20 epochs. Each epoch takes roughly 3-4 minutes on a T4 GPU.

**What to expect during training:**

- **Val F1 = 0.0 in early epochs** -- this is normal and not a bug. With only ~7 laundering edges per batch and the model outputting very low probabilities initially, nothing crosses the default threshold of 0.5. PR-AUC is the more informative metric during training because it directly measures the quality of the precision-recall trade-off for the minority class, regardless of threshold.
- **PR-AUC should climb consistently each epoch** -- if it stays flat or drops, something is wrong.
- **Loss decreasing does not necessarily mean the model is learning** -- with 99.96% legitimate edges, a model that predicts zero for everything has very low loss but is completely useless. PR-AUC is the honest metric here.

The model is saved to Google Drive whenever validation PR-AUC improves, so training can be resumed if the Colab session expires.

In [13]:
EPOCHS = 20

torch.manual_seed(SEED)
gat_baseline_model = GATModel(
    node_dim   = NODE_DIM,   # 5 (no GFP)
    edge_dim   = EDGE_DIM,   # 16 (baseline)
    hidden_dim = HIDDEN_DIM,
    num_layers = NUM_LAYERS,
    num_heads  = NUM_HEADS,
    dropout    = DROPOUT,
).to(device)

gat_baseline_model, gat_baseline_history, gat_baseline_test = run_training(
    gat_baseline_model,
    'GATv2 baseline',
    epochs          = EPOCHS,
    checkpoint_path = '/kaggle/working/Models/GAT_baseline/gat_baseline_epochs_20.pt',
)



Training GATv2 baseline
  Parameters : 39,361
  pos_weight : 8.0 (within paper range 6-8)

--- Epoch 1/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0188 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9356 | Val PR-AUC: 0.0069 | Time: 206.4s  --> New Best Model!

--- Epoch 2/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0168 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9413 | Val PR-AUC: 0.0085 | Time: 206.7s  --> New Best Model!

--- Epoch 3/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0166 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9401 | Val PR-AUC: 0.0081 | Time: 207.0s

--- Epoch 4/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0163 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9416 | Val PR-AUC: 0.0102 | Time: 207.3s  --> New Best Model!

--- Epoch 5/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


...

Result: Train Loss: 0.0150 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9424 | Val PR-AUC: 0.0140 | Time: 200.7s

--- Epoch 18/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0149 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9392 | Val PR-AUC: 0.0130 | Time: 203.3s

--- Epoch 19/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0149 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9393 | Val PR-AUC: 0.0126 | Time: 202.7s

--- Epoch 20/20 ---
  Seed edges : 4,154,429 total
  Laundering : 1,813 (0.0436%)


Result: Train Loss: 0.0148 | Val F1: 0.0000 | Val Pre: 0.0000 | Val Rec: 0.0000 | Val ROC-AUC: 0.9393 | Val PR-AUC: 0.0128 | Time: 202.6s



Final Test Results -- GATv2 baseline
  F1        : 0.0000
  Precision : 0.0000
  Recall    : 0.0000
  ROC-AUC   : 0.9505
  PR-AUC    : 0.0214


In [14]:
# Optional: load a previously saved checkpoint.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gat_baseline_model = GATModel(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, num_heads=NUM_HEADS, dropout=DROPOUT,
).to(device)

checkpoint_path = '/kaggle/working/Models/GAT_baseline/gat_baseline_epochs_20.pt'
if os.path.exists(checkpoint_path):
    gat_baseline_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print('Model loaded.')
else:
    print('Checkpoint not found.')


Model loaded.


## 9. Save Predictions for All Splits

In [ ]:
@torch.no_grad()
def score_split(model, graph, split_name):
    """
    Score all evaluated edges in one graph split.

    Returns a DataFrame with one row per evaluated transaction containing:
    - split       : which split (train / val / test)
    - src_idx     : source account integer index (from edge_index[0])
    - dst_idx     : destination account integer index (from edge_index[1])
    - timestamp   : Unix timestamp of the transaction (from edge_time)
    - score       : predicted probability of laundering
    - label       : ground truth (0=legitimate, 1=laundering)

    The combination of (src_idx, dst_idx, timestamp) uniquely identifies
    most transactions and can be used to join back to LI-Small_Trans.csv
    via the account_to_idx mapping from Data_prepration.ipynb.

    Parameters
    ----------
    model      : trained GATv2 model
    graph      : PyG Data object
    split_name : 'train', 'val', or 'test'
    """
    model.eval()
    loader, seed_edge_attr = make_loader(graph, shuffle=False)

    all_probs      = []
    all_labels     = []
    all_input_ids  = []  # positions in seed pool, used to look up metadata

    pbar = tqdm(loader, desc=f'  Scoring {split_name}', leave=False)
    for batch in pbar:
        batch     = batch.to(device)
        seed_attr = seed_edge_attr[batch.input_id.cpu()].to(device)

        logits = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            seed_attr,
        )
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch.edge_label.long().cpu().numpy())
        all_input_ids.append(batch.input_id.cpu().numpy())

    scores    = np.concatenate(all_probs)
    labels    = np.concatenate(all_labels)
    input_ids = np.concatenate(all_input_ids)  # positions into eval_mask subset

    # Recover graph-level metadata for eval edges only.
    # input_ids index into the eval subset (graph.edge_index[:, eval_mask])
    eval_edge_index = graph.edge_index[:, graph.eval_mask]  # [2, n_eval]
    eval_edge_time  = graph.edge_time[graph.eval_mask]       # [n_eval]

    src_indices = eval_edge_index[0][input_ids].numpy()  # source account idx
    dst_indices = eval_edge_index[1][input_ids].numpy()  # destination account idx
    timestamps  = eval_edge_time[input_ids].numpy()       # Unix timestamp

    df = pd.DataFrame({
        'split'    : split_name,
        'src_idx'  : src_indices,
        'dst_idx'  : dst_indices,
        'timestamp': timestamps,
        'score'    : scores,
        'label'    : labels,
    })

    print(f'  {split_name}: {len(df):,} edges  |  '
          f'{(labels==1).sum():,} laundering  |  '
          f'score range [{scores.min():.4f}, {scores.max():.4f}]  |  '
          f'mean score laund={df[df.label==1].score.mean():.4f}  '
          f'legit={df[df.label==0].score.mean():.4f}')

    return df


# Score all three splits
print('Scoring all splits...\n')
df_train = score_split(gat_baseline_model, train_graph, 'train')
df_val   = score_split(gat_baseline_model, val_graph,   'val')
df_test  = score_split(gat_baseline_model, test_graph,  'test')

# Save individually and as one combined file
SAVE_DIR = '/kaggle/working/Models/GAT_baseline/Predictions'
os.makedirs(SAVE_DIR, exist_ok=True)

df_train.to_csv(f'{SAVE_DIR}/predictions_train_gat_baseline_25_05_26.csv', index=False)
df_val.to_csv(  f'{SAVE_DIR}/predictions_val_gat_baseline_25_05_26.csv',   index=False)
df_test.to_csv( f'{SAVE_DIR}/predictions_test_gat_baseline_25_05_26.csv',  index=False)

df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_all.to_csv(f'{SAVE_DIR}/predictions_all_gat_baseline_25_05_26.csv', index=False)

print(f'\nSaved to {SAVE_DIR}/')
print(f'  predictions_train_gat_baseline_25_05_26.csv : {len(df_train):,} rows')
print(f'  predictions_val_gat_baseline_25_05_26.csv   : {len(df_val):,} rows')
print(f'  predictions_test_gat_baseline_25_05_26.csv  : {len(df_test):,} rows')
print(f'  predictions_all_gat_baseline_25_05_26.csv   : {len(df_all):,} rows')
print(f'\nSample rows from test set:')
print(df_test.head(5).to_string(index=False))

Scoring all splits...



  train: 4,154,429 edges  |  1,813 laundering  |  score range [0.0000, 0.4065]  |  mean score laund=0.0440  legit=0.0014


  val: 1,384,810 edges  |  827 laundering  |  score range [0.0000, 0.3546]  |  mean score laund=0.0268  legit=0.0014


  test: 1,384,810 edges  |  925 laundering  |  score range [0.0000, 0.4111]  |  mean score laund=0.0390  legit=0.0016

Saved to /kaggle/working/Models/GAT_baseline/Predictions/
  predictions_train_gat_baseline_25_05_26.csv : 4,154,429 rows
  predictions_val_gat_baseline_25_05_26.csv   : 1,384,810 rows
  predictions_test_gat_baseline_25_05_26.csv  : 1,384,810 rows
  predictions_all_gat_baseline_25_05_26.csv   : 6,924,049 rows

Sample rows from test set:
split  src_idx  dst_idx  timestamp    score  label
 test   635035   161142 1662653160 0.000013      0
 test   470380   296063 1662653160 0.000019      0
 test   298128   709528 1662653160 0.001047      0
 test   281432   548388 1662653160 0.000014      0
 test   692348   330214 1662653160 0.000012      0
